# Feature Engineering — Budgeted Fusion Optimization (Sections 1–2)

This development-only companion preserves the existing Training, Extraction, Combination, and Testing notebooks. It reads original v3 embeddings and leaves all CNNs and feature archives unchanged. No horizontal-flip extraction, CNN retraining, final 50K fit, or official testing is included.

**Review before execution:** all fitting is disabled by default. Importing the helper performs no fitting. Run from `tugas-7` or the repository root. The only change to the legacy Training notebook is its model-path typo; do not run that notebook as part of this experiment.

The saved baseline is **94.67% (9,467/10,000)**, not a newly reproduced result. Future official success requires **strictly >95% (at least 9,501/10,000)**. Prior individual-model official test results have already been observed. Official test features and labels are not accessed here.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "fusion_search.py").exists():
    ROOT = ROOT / "tugas-7"
if not (ROOT / "fusion_search.py").exists():
    raise FileNotFoundError("Run from the repository root or tugas-7")
sys.path.insert(0, str(ROOT.resolve()))

from fusion_search import (
    SEED, DEFAULT_POLICY, FusionExperiment, load_development,
    prepare_features, svm_evaluator, ranked,
)

# Explicit opt-ins after code review; leave False during implementation review.
RUN_BASELINE = True
RUN_SEARCH = True
EXPERIMENT_ID = "fusion_original_v4_001"
assert EXPERIMENT_ID and Path(EXPERIMENT_ID).name == EXPERIMENT_ID
EXPERIMENT_DIR = ROOT / "feature-engineering" / "experiments" / EXPERIMENT_ID

# Same ID resumes only unchanged inputs, code, policy, and package versions.
# Do not create a new ID merely to reset an exhausted budget.
print("Seed:", SEED)
print("Experiment output:", EXPERIMENT_DIR)
print("Policy:", DEFAULT_POLICY)


Seed: 30092026
Experiment output: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_original_v4_001
Policy: {'budget_seconds': 7200, 'initial_candidates': 25, 'refinement_min': 10, 'refinement_target': 15, 'refinement_max': 20, 'preferred_accuracy': 0.955, 'review_accuracy': 0.95}


## 1. Validate provenance and prepare training-only preprocessing

When baseline execution is enabled, verify the exact seeded 40K/10K development split, labels, source-block equality, and finite features. Record hashes of development arrays and current CNN files. Legacy archives do not prove which historical CNN bytes produced them; exact baseline reproduction is an additional check, not proof of that historical relationship.

Constant-channel removal and StandardScaler fit only the 40K training rows. Weighting follows scaling. Validation rows never enter an SVM fit. The experiment manifest and initial 25-candidate list are persisted before fitting. Changed inputs/code/environment require investigation and a separate experiment identity.


In [2]:
experiment = None
evaluate = None
if RUN_BASELINE:
    data, provenance = load_development(ROOT)
    X_train, X_val, block_ids = prepare_features(data)
    evaluate = svm_evaluator(
        X_train, X_val, data["y_train"], data["y_val"], block_ids,
    )
    experiment = FusionExperiment(EXPERIMENT_DIR, provenance)
else:
    print("Disabled: no features loaded, experiment outputs created, or models fitted.")


## 2. Reproduce the baseline before optimization

Parameters: weights `(1, 0.5, 1)`, `C=1`, `gamma=0.0005`, using the original embeddings and preprocessing. Require exactly **9,467 correct predictions**. Any discrepancy (including a higher score) blocks optimization. Inspect alignment, source hashes, preprocessing, and package versions before proceeding; do not change the expected score to bypass the check.

A successfully recorded baseline is reused on resume under the same experiment fingerprint and counts as one of the 25 initial configurations. Its runtime counts toward the two-hour budget.


In [3]:
if RUN_BASELINE:
    display(experiment.reproduce_baseline(evaluate))
else:
    print("Baseline has NOT been reproduced by this notebook.")


{'stop_reason': 'baseline_reproduced',
 'completed_candidates': 1,
 'compute_seconds': 6.039945678999857,
 'remaining_seconds': 7193.960054321,
 'best': {'attempts': [{'duration_seconds': 6.039945678999857,
    'started_at': '2026-10-08T00:37:55.922898+00:00',
    'status': 'completed'}],
  'candidate_id': 'a5fdcab91670e1f9689f',
  'correct': 9467,
  'duration_seconds': 6.039945678999857,
  'effective_gamma': 0.0005,
  'fit_seconds': 4.638703415000009,
  'parameters': {'C': 1.0, 'gamma': 0.0005, 'weights': [1.0, 0.5, 1.0]},
  'prediction_seconds': 1.2123570339999787,
  'status': 'completed',
  'support_vectors': 285,
  'validation_accuracy': 0.9467,
  'validation_rows': 10000},
 'official_test': 'Not evaluated. Prior individual-model test results were already observed.',
 'scope': 'Development selection only; no final 50K fit or test-ready model is saved.'}

## 3. Budgeted initial search and one local refinement

Enable `RUN_SEARCH` only after reviewing the baseline result. Start with 25 diverse joint configurations including the baseline and explicit lower-C/gamma neighbors. Freeze a single local list around the best three initial candidates: target 15, reduce toward 10 when time is limited, extend to 20 only when measured runtime leaves ample budget. No exhaustive grid or recursively repeated refinement rounds.

**Stopping rules** (checked between fits):

- Any baseline mismatch: stop and investigate.
- Validation **≥95.5%**: stop; preferred readiness for review, not guaranteed official success.
- Validation **>95% but <95.5%**: pause for review immediately. Finalization may be considered after reviewing remaining compute and generalization risk; reaching 95.5% is not mandatory. More search requires a written rationale in the separate review cell below.
- Two cumulative hours: finish the in-progress fit, then stop. Baseline, failures, and interrupted attempts count; restarting does not reset the budget.
- Completion of the initial list and one local list: stop for review; do not automatically start another experiment.
- Failed fit: record the failure and stop. Rerunning retries unfinished/failed candidates and skips completed ones.

Each attempt checkpoints parameters, score, fit/prediction/total duration, and status. Atomic JSON writes occur after each candidate; five-second heartbeats preserve approximate spent compute after abrupt termination. Only one process may use an experiment directory at a time.


In [4]:
if RUN_SEARCH:
    if experiment is None:
        raise RuntimeError("Enable RUN_BASELINE and execute its cells first")
    display(experiment.run_search(evaluate))
else:
    print("Optimization disabled. No candidate search executed.")


6a760c66307d7c5b0b26: 94.6500%, 19.7s, {'C': 0.1, 'gamma': 0.0005, 'weights': [1.0, 0.5, 1.0]}
1102639473d298c57098: 94.6700%, 9.4s, {'C': 0.3, 'gamma': 0.0005, 'weights': [1.0, 0.5, 1.0]}
bc241ea4dcf5e9c13cee: 94.7000%, 9.6s, {'C': 1.0, 'gamma': 0.0001, 'weights': [1.0, 0.5, 1.0]}
8c646eebae5a16b2f264: 94.6300%, 5.9s, {'C': 1.0, 'gamma': 0.00025, 'weights': [1.0, 0.5, 1.0]}
d8c365ddd936b1bcc4c0: 94.7000%, 12.8s, {'C': 0.3, 'gamma': 0.00025, 'weights': [1.0, 0.5, 1.0]}
d981c5ee8c1c10c6c14e: 94.5000%, 7.9s, {'C': 10.0, 'gamma': 'scale', 'weights': [1.0, 0.75, 0.75]}
aa1fee5336f0dea3ebd2: 94.6000%, 11.3s, {'C': 1.0, 'gamma': 0.0001, 'weights': [1.0, 0.75, 0.25]}
1f349fcfb63efba4d870: 94.5100%, 7.9s, {'C': 1.0, 'gamma': 'scale', 'weights': [1.0, 0.75, 0.5]}
077c70d7f5f8c708f29f: 94.5200%, 6.9s, {'C': 1.0, 'gamma': 0.00025, 'weights': [1.0, 0.5, 0.25]}
b4a29cb9dc2215c24895: 94.2800%, 16.4s, {'C': 0.1, 'gamma': 0.001, 'weights': [1.0, 1.0, 0.5]}
9eebaa0977555738ee3b: 94.4900%, 4.4s, {'C': 3

{'stop_reason': 'bounded_search_complete_review_results',
 'completed_candidates': 45,
 'compute_seconds': 496.3823316649998,
 'remaining_seconds': 6703.617668335,
 'best': {'candidate_id': '3042501f1a7f35f6111f',
  'parameters': {'C': 0.5477225575051661,
   'gamma': 0.00025,
   'weights': [1.0, 0.5, 1.0]},
  'attempts': [{'status': 'completed',
    'started_at': '2026-10-08T00:45:10.113211+00:00',
    'duration_seconds': 8.608551444000113}],
  'status': 'completed',
  'correct': 9475,
  'validation_rows': 10000,
  'fit_seconds': 5.387618492000001,
  'prediction_seconds': 3.050377387000026,
  'effective_gamma': 0.00025,
  'support_vectors': 493,
  'validation_accuracy': 0.9475,
  'duration_seconds': 8.608551444000113},
 'official_test': 'Not evaluated. Prior individual-model test results were already observed.',
 'scope': 'Development selection only; no final 50K fit or test-ready model is saved.'}

## 4. Optional continuation only after an above-95% review

Leave this disabled. If search paused above 95%, review the remaining budget, validation reuse, improvement size, and generalization risk. An affirmative review may allow the remaining bounded candidates; it never adds time, candidates beyond the planned limit, or another refinement round. Alternatively stop and discuss finalization. This notebook cannot fit a final 50K model or run official testing.


In [5]:
CONTINUE_AFTER_REVIEW = False
REVIEW_NOTE = ""  # Explain why remaining compute is justified despite validation-selection risk.
if CONTINUE_AFTER_REVIEW:
    if not RUN_SEARCH or experiment is None:
        raise RuntimeError("Run the baseline and search cells first")
    display(experiment.run_search(evaluate, review_note=REVIEW_NOTE))


## 5. Review measured results

The JSON ledger is the authoritative resumable output. A score above 95% is a development candidate, not assignment completion. Exactly 95% is below the strictly-greater target. If results remain insufficient, review them before authorizing horizontal-flip experiments or CNN retraining; neither happens automatically.


In [6]:
if experiment is not None:
    display(experiment.summary())
    rows = [
        {"candidate_id": row["candidate_id"], **row["parameters"],
         "validation_accuracy": row["validation_accuracy"],
         "correct": row["correct"], "duration_seconds": row["duration_seconds"],
         "prediction_seconds": row["prediction_seconds"]}
        for row in ranked(experiment.state)
    ]
    display(pd.DataFrame(rows))
    print("Ledger:", experiment.path)
else:
    print("No new measured results. Review implementation before execution.")


{'stop_reason': 'bounded_search_complete_review_results',
 'completed_candidates': 45,
 'compute_seconds': 496.3823316649998,
 'remaining_seconds': 6703.617668335,
 'best': {'candidate_id': '3042501f1a7f35f6111f',
  'parameters': {'C': 0.5477225575051661,
   'gamma': 0.00025,
   'weights': [1.0, 0.5, 1.0]},
  'attempts': [{'status': 'completed',
    'started_at': '2026-10-08T00:45:10.113211+00:00',
    'duration_seconds': 8.608551444000113}],
  'status': 'completed',
  'correct': 9475,
  'validation_rows': 10000,
  'fit_seconds': 5.387618492000001,
  'prediction_seconds': 3.050377387000026,
  'effective_gamma': 0.00025,
  'support_vectors': 493,
  'validation_accuracy': 0.9475,
  'duration_seconds': 8.608551444000113},
 'official_test': 'Not evaluated. Prior individual-model test results were already observed.',
 'scope': 'Development selection only; no final 50K fit or test-ready model is saved.'}

,candidate_id,C,gamma,weights,validation_accuracy,correct,duration_seconds,prediction_seconds
0,3042501f1a7f35f6111f,0.547723,0.00025,"[1.0, 0.5, 1.0]",0.9475,9475,8.608551,3.050377
1,40399026401868502c1a,1.000000,0.000158,"[1.0, 0.5, 1.0]",0.9471,9471,7.099623,2.417451
2,ea8a5b183382bc7c28e9,1.000000,0.0001,"[1.0, 0.75, 0.875]",0.9470,9470,9.264160,3.731113
3,bc241ea4dcf5e9c13cee,1.000000,0.0001,"[1.0, 0.5, 1.0]",0.9470,9470,9.642391,3.912954
4,d8c365ddd936b1bcc4c0,0.300000,0.00025,"[1.0, 0.5, 1.0]",0.9470,9470,12.814704,5.217686
5,697ec9f228bfa24be93a,0.300000,0.00025,"[1.0, 0.5, 0.875]",0.9470,9470,13.526654,5.540554
6,5979f59cfac762f4cfaf,1.000000,0.0001,"[1.0, 0.75, 1.0]",0.9469,9469,8.650529,3.385975
7,d0896d76f662476c8bce,0.300000,0.000354,"[1.0, 0.5, 1.0]",0.9469,9469,10.427685,3.645536
8,d1579be9d54db5c6461a,1.000000,0.00025,"[1.0, 0.5, 0.5]",0.9468,9468,6.524579,1.897653
9,a5fdcab91670e1f9689f,1.000000,0.0005,"[1.0, 0.5, 1.0]",0.9467,9467,6.039946,1.212357


Ledger: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_original_v4_001/search_state.json
